# 导入数据

In [1]:
import pandas as pd

In [2]:
import os
from openai import OpenAI


client = OpenAI(
    # 若没有配置环境变量，请用百炼API Key将下行替换为：api_key="sk-xxx",
    api_key="sk-e907d7f452084f2fb9715db5cdbf41c2",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

In [3]:
import pandas as pd
import random

# 读取数据
df_check = pd.read_csv("/mnt/workspace/oss/yyj_ai/fake_reviews_detection/tmp/train.csv")

# 假设label列名为'label'，如果不是请修改
# 分离正负样本
positive_samples = df_check[df_check['label'] == 1]
negative_samples = df_check[df_check['label'] == 0]

# 随机抽取各500条
sampled_positive = positive_samples.sample(n=min(250, len(positive_samples)), random_state=42)
sampled_negative = negative_samples.sample(n=min(250, len(negative_samples)), random_state=42)

# 合并并打乱顺序
df_check = pd.concat([sampled_positive, sampled_negative]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"总样本数: {len(df_check)}")
print(f"正样本(label=1): {len(df_check[df_check['label']==1])}")
print(f"负样本(label=0): {len(df_check[df_check['label']==0])}")

总样本数: 500
正样本(label=1): 250
负样本(label=0): 250


In [4]:
df_check

,user_id,review_time,image_url,review_text,label,genre,text_clean,img_local_path
0,调***儿,2021-05-07,https://img30.360buyimg.com/shaidan/s300x300_j...,收到ANESSA安热沙防晒霜后，简单的包装，有中文字体可以参考，打开看了一下，白色的乳白色的...,0,美妆,收到ANESSA安热沙防晒霜后，简单的包装，有中文字体可以参考，打开看了一下，白色的乳白色的...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
1,柚***3,10-11,https://img.alicdn.com/imgextra/i2/46116860184...,宝宝爱喝好消化好吸收,1,母婴,宝宝爱喝好消化好吸收,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
2,祝**六,2025年11月14日,//img.alicdn.com/imgextra/i1/46116860184273864...,已经回购了两年，自己抵抗力各方便都改善，明显不会感冒，真的要长期坚持吃,0,保健,已经回购了两年，自己抵抗力各方便都改善，明显不会感冒，真的要长期坚持吃,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
3,t**i,2024年11月5日,//img.alicdn.com/imgextra/i4/0/O1CN01p9U9oh1yj...,综合对比后选择了这款产品，物流安排合理，发货效率正常，包装规范，运输过程无异常，客服响应及时...,1,保健,综合对比后选择了这款产品，物流安排合理，发货效率正常，包装规范，运输过程无异常，客服响应及时...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
4,t**x,2025年11月5日,//img.alicdn.com/imgextra/i3/46116860184273811...,家里老人小孩都用着，选了这款产品，客服讲解得挺细致的，发货挺快的，用着也放心，整体看着挺靠谱...,1,保健,家里老人小孩都用着，选了这款产品，客服讲解得挺细致的，发货挺快的，用着也放心，整体看着挺靠谱...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
...,...,...,...,...,...,...,...,...
495,爱**木,2025年10月24日,//img.alicdn.com/imgextra/i4/927273224/O1CN01q...,被种草之后立马下单了这款产品，发货速度在线，客服真的挺给力的👍，包装打开那一刻就很加分✨，整...,1,保健,被种草之后立马下单了这款产品，发货速度在线，客服真的挺给力的 ，包装打开那一刻就很加分 ，整...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
496,i**u,2025年11月21日,//img.alicdn.com/imgextra/i4/46116860184273822...,给爸妈和自己买的，关注心脑血管健康。小颗粒对中老年人很友好，容易吸收。进口品牌品质放心，物流...,0,保健,给爸妈和自己买的，关注心脑血管健康。小颗粒对中老年人很友好，容易吸收。进口品牌品质放心，物流...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
497,T**6,2025年7月4日,//img.alicdn.com/imgextra/i3/46116860184273852...,收到啦收到啦。给我老妈买的。看起来挺好的，等用过以后再来追评,0,美妆,收到啦收到啦。给我老妈买的。看起来挺好的，等用过以后再来追评,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
498,匿名买家,2024年2月4日,//img.alicdn.com/imgextra/i1/0/O1CN013wiGPM1Xh...,说实话，吃了真心不错，挺高兴的，好多朋友看到我买的宝贝，都问我在哪买的，都让我把店铺告诉他们...,0,保健,说实话，吃了真心不错，挺高兴的，好多朋友看到我买的宝贝，都问我在哪买的，都让我把店铺告诉他们...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...


# Prompt

## 书写prompt

图片：image_url/img_local_path；文字：review_text；输出判断（0真实评论/1虚假评论），最后我要和真实label比较得出real和fake类别的accuracy、precision、recall、f1-score
虚假评论定义：评论内容与图片无关；文字部分存在水军/抄袭嫌疑；图片存在水军/抄袭官方图而不是自己拍/AI嫌疑

In [5]:
prompt_v3 = """
*** 任务描述
你是一名跨境电商平台的资深风控分析师，需要识别评论是否为虚假评论。请基于以下两个维度的信息进行分析：
1. 商品图片：用户上传的图片（可能为商品实拍图、官方图或其他图片）
2. 评论文字：用户对商品的文字评价

*** 虚假评论的定义（请重点参考）
虚假评论通常具有以下一个或多个特征：
1. **图文不相关**：评论内容与图片展示的商品明显不符
   - 例：评论描述"红色连衣裙很漂亮"，但图片显示的是黑色裤子
   - 例：评论讨论产品功能，但图片展示的是包装盒或无关场景

2. **文字质量异常**：
   - 模板化/通用化表述（如"很好的产品，推荐购买"，缺乏具体细节）
   - 过度夸张的赞美或诋毁，缺乏客观描述
   - 明显抄袭其他评论或官方描述（可检查重复性表述）
   - 包含营销词汇堆砌（如"限时优惠"、"必买爆款"等非用户体验词汇）

3. **图片真实性可疑**：
   - 图片为官方宣传图、网图而非用户实拍
   - 图片质量过高，有明显专业摄影或修图痕迹
   - 图片包含水印、Logo等非用户拍摄特征
   - 图片疑似AI生成（检查光影、纹理、物品边缘等不自然处）
   - 多用户使用相同图片（此条需跨评论比对，可备注怀疑）

4. **评论模式异常**：
   - 评论长度过短（如少于10字）且缺乏实质性内容
   - 评论仅包含表情符号或标点符号
   - 评论中包含联系方式、链接等违规信息

5. **跨境电商特定嫌疑**：
   - 使用机翻痕迹明显的语言
   - 明显不符合当地使用习惯的描述
   - 评价时间与商品上市时间不符（如同天上架即有多条长评）

*** 分析流程（请严格遵循）

一、图文分别分析
1. 图片分析：
   - 识别图片内容：是否为商品实拍？展示的是商品本身、包装、使用场景还是无关内容？
   - 判断图片类型：实拍图、官方图、网图还是疑似AI生成？
   - 评估图片质量：清晰度、拍摄角度、光线、背景环境等
   - 提取关键视觉特征：颜色、尺寸、品牌标识、特殊细节

2. 文字分析：
   - 提取评论核心内容：用户主要评价了什么（质量、外观、功能、服务等）
   - 评估文字质量：是否具体、有细节？还是笼统、模板化？
   - 检查语言特征：语法是否自然？有无机翻痕迹？情感表达是否真实？
   - 识别可疑模式：是否包含营销词汇、联系方式、过度重复等

二、图文一致性比对
1. 内容一致性：文字描述与图片展示是否匹配？
2. 细节一致性：文字中的具体描述（如颜色、尺寸、配件）是否在图片中得到验证？
3. 情境一致性：文字描述的使用场景是否与图片相符？

三、虚假特征综合判断
基于以上分析，判断是否存在以下情况：
1. 有明显图文不符的证据
2. 文字存在明显水军/抄袭特征
3. 图片存在非用户实拍嫌疑
4. 其他跨境电商常见的虚假评论特征

四、置信度评估
根据证据的明确程度，给出置信度评分（0-1）：
- 0.9+：有多项明确证据
- 0.7-0.9：有较强证据
- 0.5-0.7：有一定嫌疑但证据不充分
- 0.5以下：无明显虚假特征

*** 输出格式要求（必须严格遵守JSON格式）

{
  "prediction": 0 或 1,  // 0表示真实评论，1表示虚假评论
  "confidence": 0.XX,    // 置信度，0-1之间的小数
  "evidence": {
    "image_analysis": "对图片的分析描述",
    "text_analysis": "对文字的分析描述", 
    "consistency_check": "图文一致性分析",
    "key_findings": ["发现1", "发现2", "发现3"]  // 至少3个关键发现
  },
  "reasoning": "综合推理过程，解释为何得出此结论"
}

*** 示例输出（真实评论）
{
  "prediction": 0,
  "confidence": 0.85,
  "evidence": {
    "image_analysis": "图片为用户在家中的实拍图，背景可见生活物品，光线自然，无明显修图痕迹",
    "text_analysis": "评论详细描述了使用体验，提到'第一次充电需要8小时'等具体细节，语言自然",
    "consistency_check": "文字描述的'蓝色外壳'与图片中商品颜色一致，提到的'充电指示灯'在图片中可见",
    "key_findings": ["图片为真实用户实拍", "评论包含具体使用细节", "图文描述一致"]
  },
  "reasoning": "图片与文字均显示真实性特征，无模板化表述或图文不符情况，判断为真实用户评论"
}

*** 示例输出（虚假评论）
{
  "prediction": 1,
  "confidence": 0.92,
  "evidence": {
    "image_analysis": "图片为官方宣传图，包含品牌水印，背景为纯白色专业摄影棚",
    "text_analysis": "评论为通用模板'质量很好，值得购买'，缺乏具体细节，且与其他评论高度相似",
    "consistency_check": "评论未提及图片中商品的具体特征，仅使用通用评价",
    "key_findings": ["使用官方宣传图而非实拍", "评论模板化无细节", "存在抄袭其他评论嫌疑"]
  },
  "reasoning": "图片非用户实拍，评论内容模板化且与图片无关，符合虚假评论特征"
}

*** 重要提醒
1. 请优先关注图文不一致这一最直接的虚假证据
2. 对于非英语评论，注意识别机翻痕迹
3. 注意区分专业用户的高质量实拍图与官方宣传图
4. 对于边界案例，给出较低置信度并说明不确定性
5. 避免过度依赖单一特征，综合多维度判断

"""

## 定义API调用函数

In [6]:
df_check.head(3)

,user_id,review_time,image_url,review_text,label,genre,text_clean,img_local_path
0,调***儿,2021-05-07,https://img30.360buyimg.com/shaidan/s300x300_j...,收到ANESSA安热沙防晒霜后，简单的包装，有中文字体可以参考，打开看了一下，白色的乳白色的...,0,美妆,收到ANESSA安热沙防晒霜后，简单的包装，有中文字体可以参考，打开看了一下，白色的乳白色的...,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
1,柚***3,10-11,https://img.alicdn.com/imgextra/i2/46116860184...,宝宝爱喝好消化好吸收,1,母婴,宝宝爱喝好消化好吸收,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...
2,祝**六,2025年11月14日,//img.alicdn.com/imgextra/i1/46116860184273864...,已经回购了两年，自己抵抗力各方便都改善，明显不会感冒，真的要长期坚持吃,0,保健,已经回购了两年，自己抵抗力各方便都改善，明显不会感冒，真的要长期坚持吃,/mnt/workspace/oss/yyj_ai/fake_reviews_detecti...


In [ ]:
def get_fake_review_prediction(i, img_path, review_text):
    """
    分析单条评论是否为虚假评论
    
    Args:
        i: 索引/行号
        img_path: 图片本地路径
        review_text: 原始评论文字
    
    Returns:
        tuple: (i, API返回的JSON字符串)
    """
    
    # 1. 读取图片并转为base64
    try:
        with open(img_path, "rb") as image_file:
            base64_image = base64.b64encode(image_file.read()).decode('utf-8')
    except Exception as e:
        print(f"读取图片失败 {i}: {img_path}, 错误: {e}")
        return (i, '{"error": "读取图片失败"}')
    
    # 2. 调用API
    try:
        completion = client.chat.completions.create(
            model="qwen-vl-max",
            messages=[
                {
                    "role": "system",
                    "content": [{"type": "text", "text": prompt_v3}],
                },
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": f"请分析以下评论是否为虚假评论\n评论文字：{review_text}",
                        },
                        {
                            "type": "image_url",
                            "image_url": {"url": f"data:image/png;base64,{base64_image}"},
                        },
                    ],
                },
            ],
            max_tokens=500,
            temperature=0.1,  # 降低随机性，提高结果一致性
        )
        
        return (i, completion.choices[0].message.content)
        
    except Exception as e:
        print(f"API调用失败 {i}: {e}")
        return (i, '{"error": "API调用失败"}')

## 执行函数

In [8]:
import base64
import os
import json
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import time 

# 批量调用API

results = []
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {}
    
    for i, row in df_check.iterrows():
        img_path = row['img_local_path']
        review_text = row['review_text']  # 使用原始文本
        
        # 提交任务
        future = executor.submit(
            get_fake_review_prediction,
            i,
            img_path,
            review_text
        )
        futures[future] = i
    
    # 收集结果
    for future in as_completed(futures):
        i = futures[future]
        try:
            result = future.result(timeout=30)  # 30秒超时
            results.append(result)
            print(f"处理完成 {i+1}/{len(df_check)}", end="\r")
        except Exception as e:
            print(f"任务失败 {i}: {e}")
            results.append((i, '{"error": "任务执行超时或失败"}'))

# 排序结果
results = sorted(results, key=lambda x: x[0])
print(f"\n处理完成，共{len(results)}条结果")

处理完成 500/500
处理完成，共500条结果


In [12]:
results

[(0,
  '{\n  "prediction": 1,\n  "confidence": 0.87,\n  "evidence": {\n    "image_analysis": "图片为商品官方宣传风格图，背景为纯白桌面，搭配装饰性边框（如彩色圆点、爱心），整体构图精致，光线均匀，具有明显修图痕迹。产品摆放整齐，无使用痕迹，且带有‘HAPPY’等装饰文字，属于典型非用户实拍的美化图。此外，图片中可见品牌标识清晰，但无任何个人使用场景或环境特征，疑似从官网或社交媒体搬运。",\n    "text_analysis": "评论语言流畅，描述了开箱、质地、推开感、成膜速度、气味等细节，看似真实。但存在多处可疑点：1）‘简单的包装’与图片中精美展示不符；2）未提及具体防晒指数（SPF50+ PA++++）或产品型号（如ANESSA Perfect UV Sunscreen Gel），而图片中明确标注；3）‘第一次买安耐晒的防晒’与‘有中文字体可以参考’矛盾——若为首次购买，应更关注产品信息而非字体；4）语气过于中立，缺乏情绪波动，符合模板化评价特征。",\n    "consistency_check": "文字描述的‘简单包装’与图片中精心布置的高颜值展示严重不符；评论提到‘乳白色液体流动性好’，但图片未展示使用状态或涂抹效果；‘味道有点酒精味’也未在图中体现。图文内容脱节，缺乏交互验证。",\n    "key_findings": [\n      "图片为高度美化的官方风格图，非用户实拍",\n      "评论中对包装的描述与图片呈现的精致感矛盾",\n      "文字内容虽详尽但缺乏个性化细节，存在模板化嫌疑",\n      "图文之间关键信息（如产品状态、使用场景）不一致"\n    ]\n  },\n  "reasoning": "尽管评论文字包含一定细节，但其描述与图片内容存在明显矛盾：文字称包装‘简单’，而图片展示的是精心布置的商业级摄影；同时，评论未利用图片中的关键信息（如SPF50+标识）进行呼应，反而回避具体参数。结合图片的非实拍特征和评论的通用化表达，判断该评论为虚假评论，极可能'),
 (1,
  '```json\n{\n  "prediction": 1,\n  "confidence": 0.88,\n  "evidence

## 结果评估

In [10]:
import json
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def evaluate_results(results, df_check):
    """
    评估模型预测结果
    
    Args:
        results: API返回的结果列表 [(i, json_str), ...]
        df_check: 原始DataFrame，包含真实标签
    """
    
    # 1. 解析结果
    predictions = []
    true_labels = []
    
    for i, result_json in results:
        try:
            # 解析JSON
            result = json.loads(result_json)
            
            # 获取预测标签（如果有）
            if 'prediction' in result:
                pred = result['prediction']  # 0或1
                true_label = df_check.loc[i, 'label']  # 获取真实标签
                
                predictions.append(pred)
                true_labels.append(true_label)
                
        except (json.JSONDecodeError, KeyError):
            # 跳过解析失败的结果
            continue
    
    if not predictions:
        print("没有成功解析的结果！")
        return None
    
    # 转换为numpy数组
    y_true = pd.Series(true_labels)
    y_pred = pd.Series(predictions)
    
    # 2. 计算所有指标
    print(f"有效样本数: {len(y_true)}")
    print(f"真实评论(label=0): {sum(y_true==0)}")
    print(f"虚假评论(label=1): {sum(y_true==1)}")
    print(f"预测虚假评论: {sum(y_pred==1)}")
    print(f"预测真实评论: {sum(y_pred==0)}")
    
    print("\n=== 整体指标 ===")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, pos_label=1, zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred, pos_label=1, zero_division=0):.4f}")
    print(f"F1-Score:  {f1_score(y_true, y_pred, pos_label=1, zero_division=0):.4f}")
    
    # 3. 按类别计算准确率
    print("\n=== 按类别准确率 ===")
    
    # 真实评论（label=0）的准确率
    real_mask = y_true == 0
    if sum(real_mask) > 0:
        real_acc = sum((y_true == 0) & (y_pred == 0)) / sum(real_mask)
        print(f"真实评论准确率: {real_acc:.4f} ({sum((y_true==0)&(y_pred==0))}/{sum(real_mask)})")
    
    # 虚假评论（label=1）的准确率  
    fake_mask = y_true == 1
    if sum(fake_mask) > 0:
        fake_acc = sum((y_true == 1) & (y_pred == 1)) / sum(fake_mask)
        print(f"虚假评论准确率: {fake_acc:.4f} ({sum((y_true==1)&(y_pred==1))}/{sum(fake_mask)})")
    
    # 4. 混淆矩阵
    print("\n=== 混淆矩阵 ===")
    cm = confusion_matrix(y_true, y_pred)
    print(f"          预测0   预测1")
    print(f"真实0    {cm[0,0]:4d}    {cm[0,1]:4d}")
    print(f"真实1    {cm[1,0]:4d}    {cm[1,1]:4d}")
    
    return {
        'y_true': y_true,
        'y_pred': y_pred,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'recall': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'f1': f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    }

# 使用示例
metrics = evaluate_results(results, df_check)

有效样本数: 297
真实评论(label=0): 148
虚假评论(label=1): 149
预测虚假评论: 168
预测真实评论: 129

=== 整体指标 ===
Accuracy:  0.4916
Precision: 0.4940
Recall:    0.5570
F1-Score:  0.5237

=== 按类别准确率 ===
真实评论准确率: 0.4257 (63/148)
虚假评论准确率: 0.5570 (83/149)

=== 混淆矩阵 ===
          预测0   预测1
真实0      63      85
真实1      66      83


In [ ]:
# 如果需要保存详细结果
if metrics:
    # 创建包含详细信息的DataFrame
    detailed_results = []
    
    for i, result_json in results:
        try:
            result = json.loads(result_json)
            row_data = df_check.loc[i].to_dict()
            row_data.update({
                'api_prediction': result.get('prediction'),
                'api_confidence': result.get('confidence'),
                'api_error': result.get('error'),
                'is_correct': result.get('prediction') == df_check.loc[i, 'label'] if 'prediction' in result else None
            })
            detailed_results.append(row_data)
        except:
            continue
    
    results_df = pd.DataFrame(detailed_results)
    results_df.to_csv('model_predictions_details.csv', index=False, encoding='utf-8-sig')
    print(f"\n详细结果已保存到: model_predictions_details.csv")

In [11]:
def simple_evaluation(results, df_check):
    """极简版评估"""
    
    y_true = []
    y_pred = []
    
    for i, result_json in results:
        try:
            result = json.loads(result_json)
            if 'prediction' in result:
                y_true.append(df_check.loc[i, 'label'])
                y_pred.append(result['prediction'])
        except:
            continue
    
    if not y_true:
        return None
    
    # 计算指标
    metrics = {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'Recall': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'F1': f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        '真实评论准确率': sum((pd.Series(y_true)==0) & (pd.Series(y_pred)==0)) / sum(pd.Series(y_true)==0) if sum(pd.Series(y_true)==0) > 0 else 0,
        '虚假评论准确率': sum((pd.Series(y_true)==1) & (pd.Series(y_pred)==1)) / sum(pd.Series(y_true)==1) if sum(pd.Series(y_true)==1) > 0 else 0,
        '样本数': len(y_true)
    }
    
    # 打印结果
    for key, value in metrics.items():
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")
    
    return metrics

# 使用
metrics = simple_evaluation(results, df_check)

Accuracy: 0.4916
Precision: 0.4940
Recall: 0.5570
F1: 0.5237
真实评论准确率: 0.4257
虚假评论准确率: 0.5570
样本数: 297
